In [92]:
import os
import glob
import shutil
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from  vip_slap2_analysis.behavior import preprocess as ps
from vip_slap2_analysis.utils.utils import normalize_timeseries as normalize

sns.set()
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [93]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [94]:
%matplotlib notebook

In [130]:
basepath = r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics"

In [131]:
summary_path = glob.glob(os.path.join(basepath,'**summary.xlsx'))[0]
summary_df = pd.read_excel(summary_path,sheet_name='subjects')
session_df = pd.read_excel(summary_path,sheet_name='sessions')

In [132]:
process_sessions = session_df[(session_df['paradigm']=='change_detection_passive')&(session_df['session_type']!='expression_check')]
process_sessions.tail()

,session_id,subject_id,session_#,session_date,indicator1,indicator2,dmd1_depth,dmd2_depth,paradigm,session_type,...,instrument_name,instrument_id,has raster ROI?,has integration roi?,behavior_rig,quality,flags,session_dir,purpose,notes
76,826033_2026-02-18_11-57-04,826033,3,2026-02-18,iGluSnFR4,RCaMP3,50,200,change_detection_passive,familiar,...,slap2_albert,SLAP2_1,yes,no,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
77,826033_2026-02-19_13-47-57,826033,4,2026-02-19,iGluSnFR4,RCaMP3,50,200,change_detection_passive,familiar,...,slap2_albert,SLAP2_1,yes,no,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
78,826033_2026-02-21_09-23-34,826033,5,2026-02-21,iGluSnFR4,RCaMP3,50,200,change_detection_passive,familiar,...,slap2_albert,SLAP2_1,yes,no,VCO.1,good,MoCo failed in last 3min of experiment,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
79,826033_2026-02-23_10-45-21,826033,6,2026-02-23,iGluSnFR4,RCaMP3,50,200,change_detection_passive,familiar,...,slap2_albert,SLAP2_1,yes,no,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
80,826033_2026-02-24_14-14-45,826033,7,2026-02-24,iGluSnFR4,RCaMP3,50,200,change_detection_passive,familiar,...,slap2_albert,SLAP2_1,yes,no,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN


### Move extracted_files folder back into VCO1_Behavior.harp

In [98]:
old_mice = [803496,804730,804733,810196,809043,809044,809047,803121,814591,814593,825854,825855,824806]
new_mice = [826031,826032,826033]

In [100]:
# new_sess_ids = []

# for idx,row in process_sessions.iterrows():
#     sess = row['session_id']
#     sess_path = row['session_dir']
#     mouse_id = row['subject_id']    
#     if 'behavior' not in os.listdir(sess_path):
#         behavior_path = glob.glob(os.path.join(sess_path,sess,'behavior'))[0]
#     else:
#         behavior_path = glob.glob(os.path.join(sess_path,'behavior'))[0]
#     if 'extracted_files' in os.listdir(behavior_path):
#         ex_path = os.path.join(behavior_path,'extracted_files')
#         new_path = os.path.join(behavior_path,'VCO1_Behavior.harp')
#         try:
#             shutil.move(ex_path,new_path)
#             print('Successfully moved extracted data')
#         except:
#             print(f'Could not move extracted files for {sess}')
#     else:
#         print(f'No extracted data to move for {sess}')
        
#     if 'bonsai_event_log.csv' in os.listdir(behavior_path):
#         bv_path = os.path.join(behavior_path,'bonsai_event_log.csv')
#         new_path = os.path.join(behavior_path,'VCO1_Behavior.harp')
#         try:
#             shutil.move(bv_path,new_path)
#             print('Successully moved behavior data')
#         except:
#             print(f'Could not move behavior .csv for {sess}')
#     else:
#         print(f'Could not move behavior data for {sess}')


### Reprocess data

In [150]:
#Reset stimulus tables for each recording session
for idx,row in process_sessions.iterrows():
    
    sess_path = row['session_dir']
    sess = row['session_id']
    
    if 'behavior' not in os.listdir(sess_path):
        behavior_path = glob.glob(os.path.join(sess_path,sess,'behavior'))[0]
    else:
        behavior_path = glob.glob(os.path.join(sess_path,'behavior'))[0]
    pd_path = glob.glob(os.path.join(behavior_path,'**','photodiode.pkl'),recursive=True)[0]
    bon_path = glob.glob(os.path.join(behavior_path,'**','bonsai_event_log.csv'),recursive=True)[0]
    
    stim_df = pd.read_csv(bon_path)
    stim_df = stim_df[['Frame','Timestamp','Value']]
    stim_df = stim_df[stim_df['Frame']>-1].reset_index()
    stim_df.to_csv(bon_path)
    print(f'Reset data for {sess}')

Reset data for 809436_2025-07-25_13-02-10
Reset data for 809436_2025-07-28_08-04-39
Reset data for 803496_2025-07-29_13-34-35
Reset data for 803496_2025-07-30_10-05-23
Reset data for 803496_2025-07-31_09-43-28
Reset data for 803496_2025-08-01_13-22-49
Reset data for 804730_2025-07-25_14-08-35
Reset data for 804730_2025-07-28_13-57-34
Reset data for 804730_2025-07-29_14-55-04
Reset data for 804730_2025-07-30_11-11-11
Reset data for 804730_2025-07-31_11-45-27
Reset data for 804730_2025-08-01_14-22-38
Reset data for 804733_2025-07-25_15-17-00
Reset data for 804733_2025-07-28_19-00-06
Reset data for 804733_2025-07-29_16-02-24
Reset data for 804733_2025-07-30_12-59-44
Reset data for 804733_2025-07-31_13-29-01
Reset data for 804733_2025-08-01_15-20-32
Reset data for 810196_2025-07-25_16-24-20
Reset data for 810196_2025-07-28_19-59-05
Reset data for 810196_2025-07-29_17-02-41
Reset data for 810196_2025-07-31_08-28-08
Reset data for 810196_2025-07-31_14-19-46
Reset data for 810196_2025-08-01_1

In [151]:
for idx,row in process_sessions.iterrows():
    
    sess_path = row['session_dir']
    sess = row['session_id']
    
    if 'behavior' not in os.listdir(sess_path):
        behavior_path = glob.glob(os.path.join(sess_path,sess,'behavior'))[0]
    else:
        behavior_path = glob.glob(os.path.join(sess_path,'behavior'))[0]
        
    pd_path = glob.glob(os.path.join(behavior_path,'**','photodiode.pkl'),recursive=True)[0]
    bon_path = glob.glob(os.path.join(behavior_path,'**','bonsai_event_log.csv'),recursive=True)[0]
    
    try:
        print(f'Correcting alignment for {sess}')
        corrected_df, meta = ps.correct_event_log(
            bon_path,
            pd_path,
            qc_dir=Path(bon_path).parents[1],
            savepath=bon_path,                 # overwrites file
            use_piecewise_warp=True,
            insert_missing_first_stim_rows=True,  # recommended if already corrected previously
        )
        print(f"[OK] corrected via {meta.alignment_method} | slope={meta.slope:.6f} b={meta.intercept:.3f}s")
    except Exception as e:
        print(f"[FAIL] correction failed: {e}")
        plot_fail_sessions.append(session_id)
        continue

Correcting alignment for 809436_2025-07-25_13-02-10
[OK] corrected via frame_modclass_affine_anchored_edges | slope=1.000008 b=0.669s
Correcting alignment for 809436_2025-07-28_08-04-39
[OK] corrected via frame_modclass_affine_anchored_edges | slope=0.999298 b=1.109s
Correcting alignment for 803496_2025-07-29_13-34-35
[OK] corrected via frame_modclass_affine_anchored_edges | slope=1.000017 b=2.152s
Correcting alignment for 803496_2025-07-30_10-05-23
[OK] corrected via frame_modclass_affine_anchored_edges | slope=1.000003 b=2.253s
Correcting alignment for 803496_2025-07-31_09-43-28
[OK] corrected via frame_modclass_affine_anchored_edges | slope=1.000008 b=1.903s
Correcting alignment for 803496_2025-08-01_13-22-49
[OK] corrected via frame_modclass_affine_anchored_edges | slope=1.000003 b=0.696s
Correcting alignment for 804730_2025-07-25_14-08-35
[OK] corrected via frame_modclass_affine_anchored_edges | slope=1.000002 b=2.277s
Correcting alignment for 804730_2025-07-28_13-57-34
[OK] corre

In [160]:
for idx,row in process_sessions.iterrows():
    
    sess_path = row['session_dir']
    sess = row['session_id']
    
    if 'behavior' not in os.listdir(sess_path):
        behavior_path = glob.glob(os.path.join(sess_path,sess,'behavior'))[0]
    else:
        behavior_path = glob.glob(os.path.join(sess_path,'behavior'))[0]
        
    pd_path = glob.glob(os.path.join(behavior_path,'**','photodiode.pkl'),recursive=True)[0]
    bon_path = glob.glob(os.path.join(behavior_path,'**','bonsai_event_log.csv'),recursive=True)[0]
    
    stim_df = pd.read_csv(bon_path)
    
    pd_df = pd.read_pickle(pd_path)
    pd_df['time'] = pd_df.index-pd_df.index[0]
    
    
    edges = ps._get_signal_edges(pd_df['AnalogInput0'],pd_df['time'],min_edge_separation_s=1)
    
    print(sess)
    print(f'First photodiode edge: {edges[2][0]} sec')
    print(f'First BonVision frame: {stim_df["corrected_timestamps"].iloc[0]} sec \n')

809436_2025-07-25_13-02-10
First photodiode edge: 1.6480000000447035 sec
First BonVision frame: 0.6693736361636097

809436_2025-07-28_08-04-39
First photodiode edge: 1.6740159997716546 sec
First BonVision frame: 1.1088110271243603

803496_2025-07-29_13-34-35
First photodiode edge: 3.1359999999403954 sec
First BonVision frame: 2.1517688280011127

803496_2025-07-30_10-05-23
First photodiode edge: 3.2280000001192093 sec
First BonVision frame: 2.253016235065453

803496_2025-07-31_09-43-28
First photodiode edge: 2.8970240000635386 sec
First BonVision frame: 1.903344353929919

803496_2025-08-01_13-22-49
First photodiode edge: 1.6529919998720288 sec
First BonVision frame: 0.6960407890629263

804730_2025-07-25_14-08-35
First photodiode edge: 3.236991999670863 sec
First BonVision frame: 2.276527178265179

804730_2025-07-28_13-57-34
First photodiode edge: 3.1699839998036623 sec
First BonVision frame: 2.2041859331588256

804730_2025-07-29_14-55-04
First photodiode edge: 2.632000000216067 sec
Firs